# Learning Rate Scheduling

A learning-rate scheduler changes the optimizer's learning rate during training. The scheduler is stepped after the optimizer update, normally once per epoch. Metric-based schedulers receive a validation metric instead.


In [ ]:
import matplotlib.pyplot as plt
import torch

EPOCHS = 12
validation_losses = [
    1.00, 0.80, 0.70, 0.69, 0.69, 0.70,
    0.68, 0.68, 0.68, 0.67, 0.67, 0.67,
]

def create_optimizer():
    parameter = torch.nn.Parameter(torch.tensor(1.0))
    return torch.optim.SGD([parameter], lr=0.1)


## Collect Learning Rates

This helper records the learning rate at the start of each epoch. `ReduceLROnPlateau` is stepped with validation loss; the other schedulers are stepped without a metric.


In [ ]:
def collect_learning_rates(optimizer, scheduler, metric_based=False):
    learning_rates = []
    for epoch in range(EPOCHS):
        learning_rates.append(optimizer.param_groups[0]["lr"])

        # A real training loop performs backward() before this optimizer step.
        optimizer.step()
        if metric_based:
            scheduler.step(validation_losses[epoch])
        else:
            scheduler.step()

    return learning_rates


## `StepLR`

`StepLR` multiplies the learning rate by `gamma` after a fixed number of epochs. It is useful when the training plan has known decay milestones.


In [ ]:
step_optimizer = create_optimizer()
step_scheduler = torch.optim.lr_scheduler.StepLR(
    step_optimizer,
    step_size=3,
    gamma=0.5,
)
step_rates = collect_learning_rates(step_optimizer, step_scheduler)
print(step_rates)


## `ExponentialLR`

`ExponentialLR` multiplies the learning rate by `gamma` every epoch, producing smooth exponential decay.


In [ ]:
exponential_optimizer = create_optimizer()
exponential_scheduler = torch.optim.lr_scheduler.ExponentialLR(
    exponential_optimizer,
    gamma=0.85,
)
exponential_rates = collect_learning_rates(
    exponential_optimizer,
    exponential_scheduler,
)
print(exponential_rates)


## `CosineAnnealingLR`

`CosineAnnealingLR` follows a cosine curve from the initial learning rate toward `eta_min` over `T_max` epochs. It provides a gradual decay without abrupt steps.


In [ ]:
cosine_optimizer = create_optimizer()
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    cosine_optimizer,
    T_max=EPOCHS,
    eta_min=0.001,
)
cosine_rates = collect_learning_rates(cosine_optimizer, cosine_scheduler)
print(cosine_rates)


## `ReduceLROnPlateau`

`ReduceLROnPlateau` watches a metric and reduces the learning rate when improvement stalls. Unlike the other schedulers, call `step()` with the validation metric after validation finishes.


In [ ]:
plateau_optimizer = create_optimizer()
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    plateau_optimizer,
    mode="min",
    factor=0.5,
    patience=1,
)
plateau_rates = collect_learning_rates(
    plateau_optimizer,
    plateau_scheduler,
    metric_based=True,
)
print(plateau_rates)


## Visualize the Strategies

Plotting the schedules makes their behavior easier to compare. The right strategy depends on whether decay should follow fixed epochs, a smooth curve, or validation performance.


In [ ]:
scheduler_histories = {
    "StepLR": step_rates,
    "ExponentialLR": exponential_rates,
    "CosineAnnealingLR": cosine_rates,
    "ReduceLROnPlateau": plateau_rates,
}

for scheduler_name, rates in scheduler_histories.items():
    plt.plot(
        range(1, EPOCHS + 1),
        rates,
        marker="o",
        label=scheduler_name,
    )

plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("PyTorch Learning-Rate Schedulers")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## PTCA Review

Step most schedulers after the optimizer step. Pass a validation metric to `ReduceLROnPlateau`. Save and restore scheduler state when resuming training from a checkpoint.


## Save Scheduler State in a Checkpoint

A resumable training checkpoint includes model, optimizer, and scheduler state. Restoring only the optimizer does not restore scheduler history such as the last completed epoch or plateau counters.


In [ ]:
import io

checkpoint_model = torch.nn.Linear(1, 1)
checkpoint_optimizer = torch.optim.SGD(checkpoint_model.parameters(), lr=0.1)
checkpoint_scheduler = torch.optim.lr_scheduler.StepLR(
    checkpoint_optimizer, step_size=3, gamma=0.5
)

checkpoint_optimizer.step()
checkpoint_scheduler.step()

checkpoint = {
    "model_state_dict": checkpoint_model.state_dict(),
    "optimizer_state_dict": checkpoint_optimizer.state_dict(),
    "scheduler_state_dict": checkpoint_scheduler.state_dict(),
}

checkpoint_buffer = io.BytesIO()
torch.save(checkpoint, checkpoint_buffer)
checkpoint_buffer.seek(0)
restored_checkpoint = torch.load(checkpoint_buffer, weights_only=True)
print(restored_checkpoint.keys())
